# ML Foundations II: clustering, dimensionality reduction and anomaly detection

The three pages of chapter 5, run in code. Open **one** notebook, run the setup cell, then any section; each matches a page, in order.

**No API key and nothing to download.** Everything is numpy, pandas, matplotlib and scikit-learn on the firm's illustrative monthly returns (made up, not market data). The in-page versions are written in plain Python so every step can be read; here each section ends by running the same method in scikit-learn.

In [ ]:
import math
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# --- the same illustrative data and helpers the textbook's in-page cells use (ml_unsup.py) ---
MONTHS = ["Oct 24", "Nov 24", "Dec 24", "Jan 25", "Feb 25", "Mar 25", "Apr 25", "May 25", "Jun 25", "Jul 25", "Aug 25", "Sep 25",
          "Oct 25", "Nov 25", "Dec 25", "Jan 26", "Feb 26", "Mar 26", "Apr 26", "May 26", "Jun 26", "Jul 26", "Aug 26", "Sep 26"]

# Monthly returns, as fractions (0.0086 = +0.86%).
MARKET = [0.0086, -0.0166, 0.0233, 0.0585, 0.0706, 0.032, 0.01, -0.0021, -0.0093, -0.032, -0.0531, -0.0618, 0.0478, 0.0058, 0.0462, -0.0464, 0.018, 0.0074, 0.0339, -0.0012, 0.0442, 0.0462, 0.0594, -0.0718]
SIZE = [-0.0237, 0.0292, 0.0526, 0.0111, -0.0058, 0.0251, 0.0053, 0.0052, 0.028, 0.0023, -0.0331, -0.0128, 0.0135, -0.0159, 0.0087, -0.01, 0.0121, -0.0093, 0.0193, -0.0063, -0.0207, 0.0116, 0.0084, 0.0039]
VALUE = [-0.036, -0.009, 0.0265, 0.0091, -0.0278, 0.0338, 0.0129, 0.0013, -0.0093, -0.0012, 0.0146, -0.0564, -0.0069, -0.0298, 0.0129, 0.0048, 0.0027, 0.0123, 0.0442, 0.0491, -0.0294, 0.0465, 0.0049, -0.0411]
MOMENTUM = [0.0436, -0.0202, -0.0155, 0.0316, -0.0019, 0.0089, 0.0414, 0.0146, -0.0756, -0.0074, -0.0363, -0.0341, -0.0352, 0.0098, 0.0072, -0.0032, 0.0356, 0.0078, -0.0043, 0.0346, 0.0208, 0.0111, 0.0375, 0.0278]

RETURNS = {
    "DE": [0.0002, -0.0156, 0.0303, 0.0429, 0.143, 0.0623, -0.0545, 0.0035, -0.0284, -0.0655, -0.0669, -0.0805, 0.0887, -0.0112, 0.0538, -0.0353, -0.0482, -0.0356, 0.0137, 0.0464, 0.0277, 0.0372, 0.0427, -0.1365],
    "CAT": [0.0086, -0.0313, 0.0492, 0.0959, 0.1035, 0.0601, -0.0025, 0.0047, -0.0242, -0.0444, -0.0278, -0.1255, 0.0728, 0.0096, 0.063, -0.0777, 0.0231, 0.0034, 0.0224, -0.0186, -0.004, 0.0588, 0.1594, -0.1386],
    "MKT": MARKET,
}
# Month-end price before the first return, so that the last price is today's price in tools.PRICES (DE 512.40, CAT 398.15).

NAMES = ["market", "size", "value", "momentum", "DE", "CAT"]
_SERIES = {"market": MARKET, "size": SIZE, "value": VALUE, "momentum": MOMENTUM, "DE": RETURNS["DE"], "CAT": RETURNS["CAT"]}


def months(names=("market", "size", "value", "momentum")):
    """One row per month, one number per name, in percent."""
    return [[100 * _SERIES[n][i] for n in names] for i in range(len(MONTHS))]


def standardize(rows):
    """Put every column on the same scale: subtract its average, divide by its spread. Returns (z, means, spreads)."""
    d = len(rows[0])
    means = [sum(r[j] for r in rows) / len(rows) for j in range(d)]
    sds = [math.sqrt(sum((r[j] - means[j]) ** 2 for r in rows) / (len(rows) - 1)) for j in range(d)]
    return [[(r[j] - means[j]) / sds[j] for j in range(d)] for r in rows], means, sds


def dist(a, b):
    """Straight-line distance between two rows."""
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))


def kmeans(rows, k, steps=50):
    """Group rows into k clusters. Returns (centers, labels).
    Start: the first row, then repeatedly the row farthest from every centre so far (so a run is repeatable).
    Then repeat: give each row to its nearest centre, move each centre to the average of its rows."""
    centers = [list(rows[0])]
    while len(centers) < k:
        far = max(rows, key=lambda r: min(dist(r, c) for c in centers))
        centers.append(list(far))
    labels = [0] * len(rows)
    for _ in range(steps):
        new = [min(range(k), key=lambda j: dist(r, centers[j])) for r in rows]
        if new == labels and _ > 0:
            break
        labels = new
        for j in range(k):
            mine = [r for r, l in zip(rows, labels) if l == j]
            if mine:
                centers[j] = [sum(r[c] for r in mine) / len(mine) for c in range(len(rows[0]))]
    return centers, labels


def inertia(rows, centers, labels):
    """How tight the groups are: the sum of squared distances from each row to its own centre. Lower is tighter."""
    return sum(dist(r, centers[l]) ** 2 for r, l in zip(rows, labels))


def pca(rows, n=2):
    """Principal components of standardized rows: the n directions along which the months vary most.
    Returns (components, shares): each component is a list of weights, one per column;
    each share is the fraction of all the variation that direction carries."""
    z, _, _ = standardize(rows)
    d, m = len(z[0]), len(z)
    cov = [[sum(r[a] * r[b] for r in z) / (m - 1) for b in range(d)] for a in range(d)]
    total = sum(cov[i][i] for i in range(d))
    comps, shares = [], []
    for _ in range(n):
        v = [1.0 / math.sqrt(d) + 0.01 * i for i in range(d)]
        for _ in range(500):                                   # power iteration: multiply by cov until the direction settles
            w = [sum(cov[a][b] * v[b] for b in range(d)) for a in range(d)]
            norm = math.sqrt(sum(x * x for x in w))
            v = [x / norm for x in w]
        lam = sum(v[a] * sum(cov[a][b] * v[b] for b in range(d)) for a in range(d))
        big = max(range(d), key=lambda i: abs(v[i]))
        if v[big] < 0:                                         # fix the sign so the biggest weight is positive
            v = [-x for x in v]
        comps.append(v)
        shares.append(lam / total)
        cov = [[cov[a][b] - lam * v[a] * v[b] for b in range(d)] for a in range(d)]   # remove it, find the next
    return comps, shares


def project(rows, comps):
    """Each month's coordinates on the chosen components (its short description)."""
    z, _, _ = standardize(rows)
    return [[sum(w * x for w, x in zip(c, r)) for c in comps] for r in z]


def odd_scores(rows):
    """How unusual each row is: its distance from the average row, with every column on the same scale."""
    z, _, _ = standardize(rows)
    return [dist(r, [0.0] * len(r)) for r in z]


def most_odd(scores, top=3):
    """Indexes of the `top` highest scores, highest first."""
    return sorted(range(len(scores)), key=lambda i: -scores[i])[:top]

## 1 · Clustering (page 5.1)

In [ ]:
X = months()                                   # market, size, value, momentum, in percent
z, means, spreads = standardize(X)
print("typical swing of each column:", [round(s, 1) for s in spreads])

centers, labels = kmeans(z, 3)
for g in range(3):
    mine = [i for i in range(24) if labels[i] == g]
    print(f"group {g + 1}: {len(mine)} months:", ", ".join(MONTHS[i] for i in mine))
print("tightness:", round(inertia(z, centers, labels), 1))
for k in range(1, 6):
    c, l = kmeans(z, k)
    print(k, "groups: inertia", round(inertia(z, c, l), 1))

In [ ]:
# the groups on a chart: market across, momentum up, one marker shape per group
X_ = np.array(X)
for g, mk in zip(range(3), "osv"):
    pts = X_[[l == g for l in labels]]
    plt.scatter(pts[:, 0], pts[:, 3], marker=mk, s=60, label=f"group {g + 1}")
plt.xlabel("market return, %"); plt.ylabel("momentum return, %"); plt.legend(); plt.show()

# does the start matter? shuffle the order of the months and run again
import random
for seed in range(3):
    idx = list(range(24)); random.Random(seed).shuffle(idx)
    zz = [z[i] for i in idx]
    c, l = kmeans(zz, 3)
    print("shuffle", seed, "-> inertia", round(inertia(zz, c, l), 1), "sizes", sorted(l.count(g) for g in range(3)))

In [ ]:
# the same thing in scikit-learn (its starting points are random, so fix the seed)
from sklearn.cluster import KMeans
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(z)
print("scikit-learn inertia:", round(km.inertia_, 1), " sizes:", sorted(int(n) for n in np.bincount(km.labels_)))

**Try it.** Cluster on market and momentum only (`[[r[0], r[3]] for r in X]`, standardized). Do the same five sell-off months stay together? Then add Deere and Caterpillar with `months(NAMES)` and cluster again.

## 2 · Dimensionality reduction (page 5.2)

In [ ]:
X6 = months(NAMES)
components, shares = pca(X6, 3)
print("columns:", NAMES)
for i, (c, s) in enumerate(zip(components, shares), 1):
    print(f"component {i}: {100 * s:.0f}% of the variation, weights", [round(w, 2) for w in c])

P = project(X6, components[:2])
order = sorted(range(24), key=lambda i: P[i][0])
print("lowest on component 1:", [(MONTHS[i], round(P[i][0], 1)) for i in order[:3]])

In [ ]:
# each month on the first two components, coloured by the page 5.1 groups
P_ = np.array(P)
for g, mk in zip(range(3), "osv"):
    pts = P_[[l == g for l in labels]]
    plt.scatter(pts[:, 0], pts[:, 1], marker=mk, s=60, label=f"group {g + 1}")
for i in (11, 23):
    plt.annotate(MONTHS[i], P_[i])
plt.xlabel("component 1 (the market tide)"); plt.ylabel("component 2"); plt.legend(); plt.show()

In [ ]:
# the same thing in scikit-learn (a component's sign is arbitrary, so compare sizes, not signs)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
sk = PCA(n_components=3).fit(StandardScaler().fit_transform(X6))
print("scikit-learn shares:", [round(float(s), 3) for s in sk.explained_variance_ratio_])
print("ours:               ", [round(s, 3) for s in shares])

**Try it.** Run PCA on `months()` (the four benchmarks only). How much does component 1 carry, and why is it less than with six series? Then keep one component and re-cluster: which months move?

## 3 · Anomaly detection (page 5.3)

In [ ]:
scores = odd_scores(X)
print("four benchmarks, most unusual:", [(MONTHS[i], round(scores[i], 2)) for i in most_odd(scores, 5)])

scores6 = odd_scores(X6)
z6, _, _ = standardize(X6)
for i in most_odd(scores6, 3):
    big = max(range(6), key=lambda j: abs(z6[i][j]))
    print(MONTHS[i], round(scores6[i], 2), "| raw:", [round(v, 1) for v in X6[i]], "| largest column:", NAMES[big])

In [ ]:
# a chart of every month's score with a threshold
t = 2.8
plt.bar(range(24), scores6, color=["#ff5f05" if s > t else "#13294b" for s in scores6])
plt.axhline(t, linestyle="--", color="#a63d00")
plt.xticks(range(0, 24, 3), [MONTHS[i] for i in range(0, 24, 3)])
plt.ylabel("how unusual (six series)"); plt.show()
print("flagged at", t, ":", [MONTHS[i] for i in range(24) if scores6[i] > t])

In [ ]:
# the same question with scikit-learn's IsolationForest (random, so fix the seed)
from sklearn.ensemble import IsolationForest
iso = IsolationForest(random_state=0, contamination=0.15).fit(z6)
flag = iso.predict(z6) == -1
print("IsolationForest flags:", [MONTHS[i] for i in range(24) if flag[i]])
print("our distance score, top 4:", [MONTHS[i] for i in most_odd(scores6, 4)])

**Try it.** Set `t` so that exactly three months are flagged. Do `IsolationForest` and the distance score agree on those three? Where they differ, look at the raw numbers and decide which you would investigate first.